# 3 — Printed fabric in the field

This notebook gives the output for the paper section **Results → "Printed fabric in the field"**: the tables `tab:locA` and `tab:locB`, the figure `fig_field`, the relative local contrast and the empty photograph.

The two printed sheets were photographed beside three commercial garments at two locations on 2026-09-18: Location A (the generation site) and Location B (11.3 km away). `print_v1` is the pipeline's own sheet for the M81 base. **`print_v2` is not pipeline output**: it is a contrast-enhanced copy of `print_v1` (Methodology, "Fabrication"), so no result for it counts for the pipeline. The Appendix holds the code that made `print_v2`.

**Measures.** `pred_in_gt` = mean detector score inside the traced outline of the item (lower = harder to find). Each detector sees the full frame and square crops centered on the item at 1.5×, 2.5× and 4× its size; "best" (the "Max" of the tables) = the highest score over the full frame and the three crops. The items are about 1 % of the frame, where $S_\alpha$ cannot be compared (an empty prediction scores about 0.96), so the tables use `pred_in_gt`.

**Names.** `woodland_m81` is the M81 garment.

**Inputs:** the RAW photographs in `images/field_2026-09-18_two_sites/` and the traced outlines `_analysis/gt_*.png` (the same masks as `_analysis/masks/*_mask.png`; `*_outline.png` shows each outline on its photograph). Start from the project root. **Outputs:** `results/03_field_test/`.

## Research question: does the advantage survive printing?

The paper asks whether the advantage measured in software survives fabrication, real light and the material of the cloth.

*Comparison:* the analogue of notebook 2 in the field is **`print_v1`** (the site-specific M81 pattern, printed) against the **M81 garment** (its base pattern), at the same location and in the same light. `print_v2` is not pipeline output, so the answer does not use it.

*Rule:* each item has one placement only, so no statistical test is possible. A difference counts only when it is larger than the viewpoint difference that the two photographs of the same unmoved M81 drape at Location B show (same detector, same measure). Location A has no such pair, so the Location B value is used there too. That is an assumption, and a viewpoint difference is smaller than a placement difference, so this rule is lenient.

*What the answer cannot show:* the sheet and the garment also differ in fabric, ink and form (a flat sheet against a garment). So even a clear difference cannot be credited to the pattern alone.

The last cell of this notebook applies this rule.

In [ ]:
import os, sys, csv, json, shutil
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')   # let the detectors use the Apple GPU
from pathlib import Path
import numpy as np, cv2
import matplotlib.pyplot as plt
ROOT = Path.cwd()
assert (ROOT/'images').is_dir(), 'start this notebook from the project root'

# ==================== SETTINGS ====================
DEVICE    = 'mps'                               # 'mps' (Apple GPU) | 'cpu'
FRESH_RUN = True                                # True: erase the earlier outputs of this notebook first
DETECTORS = ('SINet-V2', 'DGNet')
SCALES    = [1.5, 2.5, 4.0]                     # crop side = scale x the larger side of the item's box
OUT       = ROOT/'results'/'03_field_test'

if FRESH_RUN and OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)

def load_rgb(p):
    return cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)

def conditions(img, gtb):
    """the full frame, then square crops centred on the item's box"""
    yield 'full', img, gtb
    ys, xs = np.where(gtb)
    cy, cx = (ys.min()+ys.max())//2, (xs.min()+xs.max())//2
    for s in SCALES:
        half = int(max(ys.max()-ys.min(), xs.max()-xs.min()) * s / 2)
        y0, y1 = max(0, cy-half), min(img.shape[0], cy+half)
        x0, x1 = max(0, cx-half), min(img.shape[1], cx+half)
        yield f'crop{s:g}x', img[y0:y1, x0:x1], gtb[y0:y1, x0:x1]

def score_items(items, png, gt, fig_path):
    """items = [(key..., label)]; png(item) and gt(item) give the paths. Returns one row per detector x condition."""
    rows = []
    fig, ax = plt.subplots(len(items), 5, figsize=(23, 5.2*len(items)))
    for i, it in enumerate(items):
        img = load_rgb(png(it))
        g = cv2.imread(str(gt(it)), cv2.IMREAD_GRAYSCALE)
        assert g.shape == img.shape[:2], f'{it}: GT {g.shape} vs image {img.shape[:2]}'
        gtb = g > 127
        ax[i, 0].imshow(img)
        for c in cv2.findContours(g, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]:
            ax[i, 0].plot(c[:, 0, 0], c[:, 0, 1], 'r-', lw=1.5)
        ax[i, 0].set_title(' '.join(it) + ' (traced outline)', fontsize=10)
        col = 1
        for name, det in zip(DETECTORS, (sinet, dgnet)):
            for cond, im_, g_ in conditions(img, gtb):
                pred = det.predict(im_)
                m = CODMetrics.evaluate_all(pred, g_.astype(np.float64))
                m.update(item=it, detector=name, condition=cond, pred_max=float(pred.max()),
                         pred_in_gt=float(pred[g_].mean()), pred_out_gt=float(pred[~g_].mean()))
                rows.append(m)
                if cond in ('full', 'crop2.5x'):
                    ax[i, col].imshow(im_); ax[i, col].imshow(pred, alpha=0.5, cmap='jet', vmin=0, vmax=1)
                    ax[i, col].set_title(f"{name} {cond}  in={m['pred_in_gt']:.2f}", fontsize=10); col += 1
        for a in ax[i]: a.axis('off')
    plt.tight_layout(); plt.savefig(fig_path, dpi=70, bbox_inches='tight'); plt.close(fig)   # large: saved, not shown
    print(f'overlays -> {fig_path.relative_to(ROOT)}')
    return rows

def full_and_best(rows, it, det):
    r = [x for x in rows if x['item'] == it and x['detector'] == det]
    return (next(x['pred_in_gt'] for x in r if x['condition'] == 'full'),
            max(x['pred_in_gt'] for x in r))

def write_csv(rows, path, key_names):
    cols = key_names + ['detector', 'condition', 'S_alpha', 'E_phi', 'Fw_beta', 'MAE', 'pred_max', 'pred_in_gt', 'pred_out_gt']
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=cols); w.writeheader()
        for r in rows:
            w.writerow({**dict(zip(key_names, r['item'])), **{k: r[k] for k in cols[len(key_names):]}})

fmt = lambda v: '<0.01' if v < 0.01 else f'{v:.2f}'

## Code

The cells of this section hold all the code that the steps below use. Each code cell is one module, and the text above it names the section of the paper that it implements. Run this section before the steps.

In [ ]:
# A cell with %%module on its first line holds one module of the code. The cell runs as its
# own Python module, so its names do not mix with the names of the notebook (both can have OUT,
# for example). Other cells get a module with a usual import, for example `import adversarial_eval as rv`.
import sys, types, linecache
from IPython.core.magic import register_cell_magic

class CellSource:
    """Gives the cell text to linecache again, so that tracebacks and TensorFlow can read it."""
    def __init__(self, text):
        self.text = text

    def get_source(self, name):
        return self.text

@register_cell_magic
def module(line, cell):
    name = line.strip()
    m = types.ModuleType(name)
    m.__file__ = f'%%module {name}'                    # the file name in tracebacks
    m.__loader__ = CellSource(cell)
    linecache.cache[m.__file__] = (len(cell), None, cell.splitlines(True), m.__file__)
    sys.modules[name] = m
    exec(compile(cell, m.__file__, 'exec', dont_inherit=True), m.__dict__)

### Color calibration — Methodology, "Data Acquisition"

Each RAW file is decoded with rawpy to linear color with sRGB primaries, using the white balance of the chart photograph. The 24 chart patches are found automatically. After the gray patches are scaled to the reference white, a 3×3 matrix is fitted in linear sRGB from the measured patches to the published chart values, and it corrects every photograph. Each photograph is then scaled so that its median brightness matches the same decode with LibRaw's automatic brightness.

The scene photographs are decoded at half size (`half_size=True`): LibRaw joins each 2×2 Bayer block into one pixel and does no demosaicing. Only the chart photograph is decoded at full size with AHD demosaicing.

In [ ]:
%%module calibration
"""Color calibration of the RAW photographs with one ColorChecker chart.

1. The chart photograph is decoded to linear sRGB with the chart's white balance, and the
   24 patches are found automatically.
2. The gray patches are scaled to the reference white, and a 3x3 matrix is fitted in linear
   sRGB from the measured patches to the published chart values.
3. Each other photograph is decoded with the same white balance, corrected with the matrix,
   and scaled so that its median brightness is equal to the same decode with LibRaw's
   automatic brightness. The result is an 8-bit sRGB PNG.
"""
import glob, json, os
from pathlib import Path

import numpy as np, rawpy, cv2

ROOT = Path.cwd()          # project root (the notebook starts here)
N_PATCHES = 24
# colour-science reference data for the chart (the first name that exists is used)
REF_DATASET_CANDIDATES = ["ColorChecker24 - After November 2014",
                          "ColorChecker24 - Before November 2014",
                          "ColorChecker 2005"]


def _fix_wb(wb):
    """Bayer sensors often give 0 for the second green; use the first green there."""
    wb = list(wb)
    if len(wb) >= 4 and wb[3] == 0:
        wb[3] = wb[1]
    return wb


def decode_chart(dng_path):
    """Chart photograph -> linear sRGB in [0, 1] at full size, and its camera white balance."""
    with rawpy.imread(dng_path) as raw:
        cam_wb = _fix_wb(raw.camera_whitebalance)
        rgb16 = raw.postprocess(gamma=(1, 1), no_auto_bright=True, output_bps=16,
                                output_color=rawpy.ColorSpace.sRGB,
                                demosaic_algorithm=rawpy.DemosaicAlgorithm.AHD,
                                highlight_mode=rawpy.HighlightMode.Clip, user_wb=cam_wb)
    return rgb16.astype(np.float32) / 65535.0, cam_wb


def decode(dng, wb, auto_bright, half=True):
    """Scene photograph -> linear sRGB in [0, 1] with the white balance `wb`.
    half=True gives LibRaw's half-size output (each 2x2 Bayer block becomes one pixel)."""
    with rawpy.imread(dng) as raw:
        rgb = raw.postprocess(gamma=(1, 1), no_auto_bright=(not auto_bright), output_bps=16,
                              output_color=rawpy.ColorSpace.sRGB,
                              demosaic_algorithm=rawpy.DemosaicAlgorithm.AHD,
                              highlight_mode=rawpy.HighlightMode.Clip, user_wb=_fix_wb(wb), half_size=half)
    return rgb.astype(np.float32) / 65535.0


def lin_to_lab(lin):
    """Linear sRGB -> CIE Lab (D65)."""
    import colour
    return colour.XYZ_to_Lab(colour.sRGB_to_XYZ(np.clip(np.asarray(lin, float), 0, 1), apply_cctf_decoding=False))


def reference_swatches():
    """The 24 published chart colors as linear sRGB (D65), their Lab values, and the dataset name."""
    import colour
    name = next(n for n in REF_DATASET_CANDIDATES if n in colour.CCS_COLOURCHECKERS)
    ds = colour.CCS_COLOURCHECKERS[name]
    xyY = np.array(list(ds.data.values()))
    lin = colour.XYZ_to_sRGB(colour.xyY_to_XYZ(xyY), illuminant=ds.illuminant, apply_cctf_encoding=False)
    lin = np.clip(lin, 0.0, 1.0).astype(np.float64)
    return lin, lin_to_lab(lin), name


def detect_swatches(lin):
    """Find the chart and return its 24 patch colors as linear sRGB (24 x 3).
    The detector expects a gamma-encoded image, so the image is encoded first and the
    patch colors are decoded back to linear light."""
    from colour_checker_detection import detect_colour_checkers_segmentation
    a = np.clip(lin, 0, 1)
    enc = np.where(a <= 0.0031308, 12.92 * a, 1.055 * np.power(a, 1 / 2.4) - 0.055)
    res = detect_colour_checkers_segmentation(enc.astype(np.float32), additional_data=True)
    if not res:
        raise RuntimeError('ColorChecker not found')
    sw = np.asarray(res[0].swatch_colours, float)
    assert sw.shape == (N_PATCHES, 3), sw.shape
    sw = np.clip(sw, 0, 1)
    return np.where(sw <= 0.04045, sw / 12.92, ((sw + 0.055) / 1.055) ** 2.4)


def luma(x):
    return 0.2126 * x[..., 0] + 0.7152 * x[..., 1] + 0.0722 * x[..., 2]


def save8(path, lin):
    """Linear sRGB -> 8-bit sRGB PNG."""
    a = np.clip(lin, 0, 1)
    enc = np.where(a <= 0.0031308, 12.92 * a, 1.055 * np.power(a, 1 / 2.4) - 0.055)
    cv2.imwrite(str(path), (np.clip(enc * 255 + 0.5, 0, 255).astype(np.uint8))[..., ::-1])


def calibrate(site, chart, out, exclude=()):
    """Calibrate every DNG in `site` with the chart photograph `chart` and write PNGs to `out`.
    `site` and `out` are paths from the project root. The chart and the frames in `exclude`
    get no PNG."""
    site_dir, out_dir = ROOT / site, ROOT / out
    out_dir.mkdir(parents=True, exist_ok=True)
    exclude = set(exclude) | {chart}

    ref_lin, ref_lab, ds = reference_swatches()
    chart_lin, chart_wb = decode_chart(str(site_dir / f"{chart}.DNG"))
    sw = detect_swatches(chart_lin)
    print(f"chart {chart}: detected via auto-detect, dataset {ds}")

    # scale the gray patches (18..23) to the reference white, then fit M so that ref = sw @ M
    s = float(np.median(ref_lin[18:24]) / max(np.median(sw[18:24]), 1e-6))
    M, *_ = np.linalg.lstsq(sw * s, ref_lin, rcond=None)
    apply = lambda img: np.clip(img.reshape(-1, 3) @ M, 0, 1).reshape(img.shape)
    import colour
    de = np.asarray(colour.delta_E(lin_to_lab(np.clip(apply((sw * s)[None])[0], 0, 1)), ref_lab, method="CIE 2000"))
    print(f"gain-normalised linear3x3 CCM: chart dE2000 mean={de.mean():.2f} max={de.max():.2f}")

    frames = sorted(Path(p).stem for p in glob.glob(str(site_dir / "*.DNG")) if os.path.getsize(p) > 0)
    frames = [f for f in frames if f not in exclude]
    for f in frames:
        dng = str(site_dir / f"{f}.DNG")
        corrected = np.clip(apply((decode(dng, chart_wb, auto_bright=False) * s)[None])[0], 0, 1)
        target = float(np.median(luma(decode(dng, chart_wb, auto_bright=True))))
        k = target / max(float(np.median(luma(corrected))), 1e-5)
        save8(out_dir / f"{f}.png", np.clip(corrected * k, 0, 1))
        print(f"  {f}.png")
    json.dump({"site": str(site), "chart": chart, "method": "gainnorm_linear3x3+exposure_norm",
               "chart_dE_mean": float(de.mean()), "frames": frames, "excluded": sorted(exclude)},
              open(out_dir / "calibration_manifest.json", "w"), indent=1)
    print(f"\ncalibrated {len(frames)} frames -> {out_dir}")

### Scores — Evaluation, "Digital Evaluation"

The paper uses the structure measure $S_\alpha$ (Fan et al. 2017), which compares the whole prediction map with the target mask. A lower $S_\alpha$ means that the detector found the target less well. The other scores ($E_\phi$, $F^w_\beta$, MAE) are stored in the result files but are not used in the paper.

In [ ]:
%%module cod_metrics
"""Detector scores. S_alpha (structure measure, Fan et al. 2017) is the score the paper uses.

    S_alpha - structure measure              (Fan et al., ICCV 2017)
    E_phi   - enhanced-alignment measure     (Fan et al., IJCAI 2018)
    Fw_beta - weighted F-measure             (Margolin et al., CVPR 2014)
    MAE     - mean absolute error

The functions are line-for-line ports of the MATLAB files in models/SINet-V2/eval/
(S_object.m, S_region.m, Enhancedmeasure.m, original_WFb.m). MATLAB std and var use N-1,
so every variance here uses ddof=1.
"""
import numpy as np
import cv2
from scipy.ndimage import distance_transform_edt as _bwdist

_EPS = np.finfo(np.float64).eps


def _matlab_round(v: float) -> int:
    """MATLAB round(): half away from zero. Python's round() is half to even."""
    return int(np.floor(v + 0.5)) if v >= 0 else int(np.ceil(v - 0.5))


class CODMetrics:
    # ------------------------------------------------------------------ MAE
    @staticmethod
    def mae(pred: np.ndarray, gt: np.ndarray) -> float:
        return float(np.mean(np.abs(pred.astype(np.float64) - gt.astype(np.float64))))

    # -------------------------------------------------------------- S-measure
    @staticmethod
    def _object(pred_region: np.ndarray) -> float:
        """S_object.m: score = 2x / (x^2 + 1 + sigma_x + eps)."""
        if pred_region.size == 0:
            return 0.0
        x = float(pred_region.mean())
        sigma_x = float(pred_region.std(ddof=1)) if pred_region.size > 1 else 0.0
        return float(2.0 * x / (x * x + 1.0 + sigma_x + _EPS))

    @classmethod
    def s_object(cls, pred: np.ndarray, gt: np.ndarray) -> float:
        fg = np.where(gt, pred, 0.0)
        bg = np.where(gt, 0.0, 1.0 - pred)
        u = float(gt.mean())
        return u * cls._object(fg[gt]) + (1.0 - u) * cls._object(bg[~gt])

    @staticmethod
    def _ssim(pred: np.ndarray, gt: np.ndarray) -> float:
        """S_region.m's ssim helper, sample variance (N-1)."""
        n = pred.size
        if n == 0:
            return 0.0
        if n == 1:
            # S_region.m: for N==1 all three variances are exactly 0, so alpha==0 and
            # beta==0 and the function takes its Q=1.0 branch regardless of the values.
            return 1.0
        x, y = float(pred.mean()), float(gt.mean())
        sx2 = float(((pred - x) ** 2).sum() / (n - 1 + _EPS))
        sy2 = float(((gt - y) ** 2).sum() / (n - 1 + _EPS))
        sxy = float(((pred - x) * (gt - y)).sum() / (n - 1 + _EPS))
        a = 4.0 * x * y * sxy
        b = (x * x + y * y) * (sx2 + sy2)
        if a != 0.0:
            return float(a / (b + _EPS))
        return 1.0 if (a == 0.0 and b == 0.0) else 0.0

    @classmethod
    def s_region(cls, pred: np.ndarray, gt: np.ndarray) -> float:
        """S_region.m: 4 quadrants about the GT centroid, area-weighted SSIM."""
        h, w = gt.shape
        total = float(gt.sum())
        if total == 0:
            return 0.0
        gtf = gt.astype(np.float64)
        # centroid, MATLAB 1-based
        # MATLAB round() is half-AWAY-from-zero; Python's round() is half-to-EVEN.
        # They disagree on every exact .5 centroid with an even integer part, which
        # shifts the quadrant split by one pixel (audited 2026-09-20: worst case 0.50
        # on 2x2 objects, but 0/180 of this pipeline's silhouette masks ever tie).
        X = _matlab_round(float((np.arange(1, w + 1) * gtf.sum(axis=0)).sum() / total))
        Y = _matlab_round(float((np.arange(1, h + 1) * gtf.sum(axis=1)).sum() / total))
        # DELIBERATE DEVIATION: S_region.m does not clamp, so a centroid on the last
        # row/column gives an empty quadrant, mean2([]) = NaN, and the whole measure
        # returns NaN. We clamp to keep the pipeline finite. Verified never to bind on
        # this corpus (the centroid is interior for every mask used here).
        X = min(max(X, 1), w - 1)
        Y = min(max(Y, 1), h - 1)
        quads = [(slice(0, Y), slice(0, X)), (slice(0, Y), slice(X, w)),
                 (slice(Y, h), slice(0, X)), (slice(Y, h), slice(X, w))]
        area = float(w * h)
        w1 = (X * Y) / area
        w2 = ((w - X) * Y) / area
        w3 = (X * (h - Y)) / area
        w4 = 1.0 - w1 - w2 - w3
        return float(sum(wi * cls._ssim(pred[q].astype(np.float64), gtf[q])
                         for wi, q in zip((w1, w2, w3, w4), quads)))

    @classmethod
    def s_measure(cls, pred: np.ndarray, gt: np.ndarray, alpha: float = 0.5) -> float:
        """StructureMeasure.m"""
        pred = pred.astype(np.float64)
        gtb = gt > 0.5
        y = float(gtb.mean())
        if y == 0.0:
            return float(1.0 - pred.mean())
        if y == 1.0:
            return float(pred.mean())
        q = alpha * cls.s_object(pred, gtb) + (1.0 - alpha) * cls.s_region(pred, gtb)
        return float(max(q, 0.0))

    # -------------------------------------------------------------- E-measure
    @staticmethod
    def _enhanced(binary_pred: np.ndarray, gt: np.ndarray) -> float:
        """Enhancedmeasure.m. Its first line is `FM = logical(FM)`, so the reference
        takes a BINARY prediction. main.m:97-102 binarizes at min(2*mean(cam), 1)
        before calling it. Feeding a continuous sigmoid map here (as the pre-2026-09-20
        code did) is not what the reference computes."""
        pred = binary_pred.astype(np.float64)
        gtf = (gt > 0.5).astype(np.float64)
        h, w = gtf.shape
        s = gtf.sum()
        if s == 0.0:
            enhanced = 1.0 - pred
        elif s == gtf.size:
            enhanced = pred.copy()
        else:
            d_pred = pred - pred.mean()
            d_gt = gtf - gtf.mean()
            align = 2.0 * (d_gt * d_pred) / (d_gt * d_gt + d_pred * d_pred + _EPS)
            enhanced = ((align + 1.0) ** 2) / 4.0
        return float(enhanced.sum() / (w * h - 1 + _EPS))

    @classmethod
    def e_measure(cls, pred: np.ndarray, gt: np.ndarray) -> float:
        """Adaptive E-measure, main.m:97-102: binarize at min(2*mean(pred), 1)."""
        p = pred.astype(np.float64)
        thr = min(2.0 * float(p.mean()), 1.0)
        return cls._enhanced(p > thr, gt)

    # ------------------------------------------------------------- weighted F
    # The distance transform and the importance map depend ONLY on the GT mask, not on
    # the prediction, but the reference recomputes them per call. In this pipeline the
    # same mask is scored 8 times (2 conditions x 2 detectors x 2 normalisations), so a
    # tiny cache keyed on the mask cuts the dominant cost without changing any number.
    _wfb_cache: dict = {}

    @classmethod
    def _wfb_geometry(cls, gtb: np.ndarray):
        key = (gtb.shape, gtb.tobytes())
        hit = cls._wfb_cache.get(key)
        if hit is None:
            dst, idx = _bwdist(~gtb, return_indices=True)
            b = np.where(gtb, 1.0, 2.0 - np.exp(np.log(1 - 0.5) / 5.0 * dst))
            # int32 indices (exact for any real image) halve the entry to ~36 MB,
            # and the bound is 2 because the access pattern is 8 consecutive calls on one
            # mask (2 conditions x 2 detectors x 2 normalisations) before the mask changes.
            hit = (idx.astype(np.int32), b)   # int32 indices are exact; b stays float64
            if len(cls._wfb_cache) >= 2:
                cls._wfb_cache.clear()
            cls._wfb_cache[key] = hit
        return hit

    @classmethod
    def weighted_f_measure(cls, pred: np.ndarray, gt: np.ndarray, beta: float = 1.0) -> float:
        """original_WFb.m (Margolin et al. 2014)."""
        pred = pred.astype(np.float64)
        gtb = gt > 0.5
        if not gtb.any():
            return 0.0
        e = np.abs(pred - gtb.astype(np.float64))
        idx, b = cls._wfb_geometry(gtb)
        # Et: pixel dependency - background error inherits its nearest GT pixel's error
        et = e.copy()
        et[~gtb] = e[idx[0][~gtb], idx[1][~gtb]]
        k = cv2.getGaussianKernel(7, 5.0)
        ea = cv2.filter2D(et, -1, k @ k.T, borderType=cv2.BORDER_CONSTANT)
        min_e_ea = e.copy()
        sel = gtb & (ea < e)
        min_e_ea[sel] = ea[sel]
        ew = min_e_ea * b          # b: pixel importance, from the cached geometry
        tpw = float(gtb.sum()) - float(ew[gtb].sum())
        fpw = float(ew[~gtb].sum())
        r = 1.0 - float(ew[gtb].mean())
        p = tpw / (tpw + fpw + _EPS)
        b2 = beta ** 2
        return float((1.0 + b2) * r * p / (b2 * p + r + _EPS))

    # ------------------------------------------------------------------ all
    @classmethod
    def evaluate_all(cls, pred: np.ndarray, gt: np.ndarray) -> dict:
        p = pred.astype(np.float64)
        g = gt.astype(np.float64)
        if p.max() > 1.01:
            p /= 255.0
        if g.max() > 1.01:
            g /= 255.0
        # main.m:89 min-max stretches every prediction to [0,1] before scoring.
        # That convention makes a uniformly-near-zero map (a total miss) look like a
        # confident detection, which is exactly the wrong behaviour for camouflage
        # evaluation, so the UNNORMALISED values are primary here and the benchmark
        # convention is reported alongside for comparability.
        rng = float(p.max() - p.min())
        pn = (p - p.min()) / rng if rng > 1e-12 else np.zeros_like(p)
        return {
            'MAE':          cls.mae(p, g),
            'S_alpha':      cls.s_measure(p, g),
            'E_phi':        cls.e_measure(p, g),
            'Fw_beta':      cls.weighted_f_measure(p, g),
            'S_alpha_norm': cls.s_measure(pn, g),
            'E_phi_norm':   cls.e_measure(pn, g),
            'Fw_beta_norm': cls.weighted_f_measure(pn, g),
        }

### The two detectors — Evaluation, "Digital Evaluation"

SINet-V2 and DGNet act as fixed judges. Both are used exactly as published, with no retraining; their code and weights are in `models/`.

In [ ]:
%%module detectors
"""The two camouflaged object detectors, used as published with no retraining.

    SINet-V2  Fan et al. 2022, IEEE TPAMI   https://github.com/GewelsJI/SINet-V2
    DGNet     Ji et al. 2023, MIR           https://github.com/GewelsJI/DGNet (EfficientNet-B4)

Each detector takes an RGB uint8 image and returns a float32 map in [0, 1] at the image
size: the sigmoid of its output after a resize of the input to 352 x 352.

torch loads only when build_detectors() runs. When torch and torchvision load before the
k-means palette step in the same process, the palette changes (3 pixels of the M81
pattern in a test on 2026-09-28), so the import must stay inside the function.
"""
import sys, warnings
from pathlib import Path

import numpy as np

ROOT = Path.cwd()          # project root (the notebook starts here)
SINET_WEIGHTS = ROOT / 'models' / 'SINet-V2' / 'snapshot' / 'SINet_V2' / 'Net_epoch_best.pth'
DGNET_WEIGHTS = ROOT / 'models' / 'DGNet' / 'lib_pytorch' / 'snapshot' / 'DGNet.pth'


def _import_repo(repo_dir):
    """Both repositories have a top-level package `lib`. Remove the one that is loaded and
    put this repository first on the path."""
    for mod in [m for m in list(sys.modules) if m == 'lib' or m.startswith('lib.')]:
        del sys.modules[mod]
    sys.path.insert(0, str(repo_dir))


class Detector:
    def __init__(self, model, device, pick):
        import torch
        import torchvision.transforms as T
        self.torch, self.model, self.device, self.pick = torch, model.to(device).eval(), device, pick
        self.transform = T.Compose([T.ToPILImage(), T.Resize((352, 352)), T.ToTensor(),
                                    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

    def predict(self, image):
        torch = self.torch
        h, w = image.shape[:2]
        x = self.transform(image).unsqueeze(0).to(self.device)
        with torch.no_grad():
            pred = torch.sigmoid(self.pick(self.model(x)))
            pred = torch.nn.functional.interpolate(pred, size=(h, w), mode='bilinear', align_corners=True)
        return pred.squeeze().cpu().numpy().astype(np.float32)


def build_detectors(device):
    """Returns (SINet-V2, DGNet) on `device` ('mps' or 'cpu')."""
    import torch
    device = torch.device(device)

    _import_repo(SINET_WEIGHTS.resolve().parents[2])
    from lib.Network_Res2Net_GRA_NCD import Network
    sinet = Network(channel=32, imagenet_pretrained=False)
    sinet.load_state_dict(torch.load(str(SINET_WEIGHTS), map_location=device, weights_only=False))
    print(f'[SINetV2] Official model loaded — device={device}')

    _import_repo(DGNET_WEIGHTS.resolve().parents[1])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        from lib.DGNet import DGNet
    dgnet = DGNet(channel=64, arc='EfficientNet-B4', M=[8, 8, 8], N=[4, 8, 16])
    # strict=False: the two extra keys are the classification head of the backbone, which DGNet does not use
    missing, _ = dgnet.load_state_dict(torch.load(str(DGNET_WEIGHTS), map_location=device, weights_only=False), strict=False)
    assert not missing, f'DGNet: {len(missing)} missing keys'
    print(f'[DGNet] Model loaded (EfficientNet-B4, channel=64) — device={device}')

    # SINet-V2 returns several maps, the last is the prediction; DGNet returns (prediction, gradient map)
    return (Detector(sinet, device, lambda out: out[-1] if isinstance(out, (list, tuple)) else out),
            Detector(dgnet, device, lambda out: out[0] if isinstance(out, (tuple, list)) else out))

### Tables for the paper

Each function reads a result file and returns one table of the paper in LaTeX. Nothing in a table is typed by hand.

In [ ]:
%%module paper_tables
"""LaTeX tables for the paper, in the format of the manuscript, made from the saved result files.

Each function reads a CSV that a notebook wrote, returns the LaTeX of one table, and
writes it to `out` when `out` is given. Nothing in a table is typed by hand.
"""
import csv
from collections import defaultdict
from pathlib import Path

import numpy as np
from scipy import stats

STYLES = ['multicam', 'woodland', 'realtree']
NAME = {'multicam': 'MultiCam', 'woodland': 'M81', 'realtree': 'Realtree'}   # 'woodland' is the M81 base
DETECTORS = ('SINet-V2', 'DGNet')


def _holm(ps):
    order = sorted(range(len(ps)), key=lambda i: ps[i])
    adj, prev = [0.0] * len(ps), 0.0
    for rank, i in enumerate(order):
        prev = max(prev, min(1.0, (len(ps) - rank) * ps[i]))
        adj[i] = prev
    return adj


def _write(tex, out):
    if out:
        out = Path(out)
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_text(tex + '\n')
    return tex


def quant_table(table_csv, out=None):
    """Table tab:quant from colour_limit/table_quant_ablation.csv (notebook 1)."""
    rows = [r for r in csv.DictReader(open(table_csv)) if r['metric'] == 'S_alpha']
    get = lambda st, det, c: next(r for r in rows if r['style'] == st and r['detector'] == det and r['contrast'] == c)
    cells = [(st, det) for st in STYLES for det in DETECTORS]
    hp = dict(zip(cells, _holm([float(get(st, det, 'dq_nst-dq_nonst')['p_two_sided']) for st, det in cells])))
    hw = dict(zip(cells, _holm([float(get(st, det, 'k5-kfull')['p_two_sided']) for st, det in cells])))
    ho = dict(zip(cells, _holm([float(get(st, det, 'nonst_k5-nonst_kfull')['p_two_sided']) for st, det in cells])))
    star = lambda v, p: f'${v:+.3f}^{{*}}$' if p < 0.05 else f'${v:+.3f}$'
    body = []
    for st in STYLES:
        for det in DETECTORS:
            w, wo, d = (get(st, det, c) for c in ('k5-kfull', 'nonst_k5-nonst_kfull', 'dq_nst-dq_nonst'))
            dv = float(d['delta'])
            # bold = reduction significant after Holm; * = dq significant after Holm (per column)
            cell = rf'$\mathbf{{{dv:+.3f}}}$' if dv < 0 and hp[(st, det)] < 0.05 else f'${dv:+.3f}$'
            body.append(f"{NAME[st]:8} & {det:8} & {star(float(w['delta']), hw[(st, det)])} & "
                        f"{star(float(wo['delta']), ho[(st, det)])} & {cell} \\\\")
    return _write(r'''\begin{table}[t]
\centering
\caption{The cost of the five-color limit. Each row is named after the base pattern; the values are about the site-specific pattern made from it. $\Delta_q$ is the $S_\alpha$ score
with five colors minus the score with no color limit; a positive value means
five colors make the pattern easier to find. Each value is the mean of the 33
scene-level differences. ``$\Delta_q$ with'' uses the
style-transfer pattern; ``$\Delta_q$ without'' repeats the test on the base pattern
alone, at the same image size. The difference is with minus without; a negative
value means style transfer reduces the cost. Two-sided Wilcoxon tests over 33
scenes, Holm correction over the six cells of each column: an asterisk marks a
significant $\Delta_q$, bold a significant reduction.}
\label{tab:quant}
\small
\setlength{\tabcolsep}{4pt}
\begin{tabular}{llrrr}
\hline
Base & Detector & $\Delta_q$ with & $\Delta_q$ without & difference \\
\hline
''' + '\n'.join(body) + r'''
\hline
\end{tabular}
\end{table}''', out)


def base_table(results_csv, out=None):
    """Table tab:base from 02_adversarial_test/IMG_4171_cal/results.csv (notebook 2)."""
    agg = defaultdict(list)
    for r in csv.DictReader(open(results_csv)):
        agg[(r['bg'], r['style'], r['detector'], r['shape'], r['condition'])].append(float(r['S_alpha']))
    bgs = sorted({k[0] for k in agg}); shapes = sorted({k[3] for k in agg})
    cells = [(st, det) for st in STYLES for det in DETECTORS]
    D = {(st, det): np.array([np.mean([np.mean(agg[(b, st, det, sh, 'site')]) - np.mean(agg[(b, st, det, sh, 'generic')])
                                       for sh in shapes]) for b in bgs]) for st, det in cells}
    H = dict(zip(cells, _holm([stats.wilcoxon(D[c]).pvalue for c in cells])))
    holm_note = ("``better'' and ``worse'' are significant after Holm correction over the six tests; "
                 "``no difference'' is not")
    body = []
    for st, det in cells:
        d = D[(st, det)]
        res = ('no difference' if H[(st, det)] >= 0.05 else
               'better' if d.mean() < 0 else r'\textbf{worse}')
        hp_ = H[(st, det)]
        ptxt = '$<0.001$' if hp_ < 0.001 else f'${hp_:.2f}$'
        body.append(f"{NAME[st]:8} & {det:8} & ${d.mean():+.3f}$ & {int((d < 0).sum())}/{len(d)} & {ptxt} & {res} \\\\")
    return _write(r'''\begin{table}[t]
\centering
\caption{Each site-specific pattern against the base pattern it was made from.
Each row is named after the base pattern. $\Delta S_\alpha$ is the mean of the 33 scene-level differences, site-specific
minus base, so a negative value means the site-specific pattern is harder to find. The base is downsampled to the size of the
site-specific pattern. ``Wins'' counts how many of the 33
scenes favor it. Tests are two-sided Wilcoxon signed-rank; ''' + holm_note + r'''.}
\label{tab:base}
\footnotesize
\setlength{\tabcolsep}{3pt}
\begin{tabular}{llrrrl}
\hline
Base & Detector & $\Delta S_\alpha$ & Wins & Holm $p$ & Result \\
\hline
''' + '\n'.join(body) + r'''
\hline
\end{tabular}
\end{table}''', out)


FIELD_LABEL = {'print_v1': r'\texttt{print\_v1} (pipeline)', 'print_v2': r'\texttt{print\_v2}',
               'multicam_ocp': 'MultiCam', 'woodland_m81': 'M81, view 1', 'woodland_m81_b': 'M81, view 2',
               'hunting_realtree': 'Realtree'}
FIELD_ORDER = ['print_v1', 'print_v2', 'multicam_ocp', 'woodland_m81', 'woodland_m81_b', 'hunting_realtree']


def location_table(results_csv, site, out=None):
    """Table tab:locA (site1) or tab:locB (site2) from 03_field_test/two_sites/results.csv (notebook 3)."""
    rows = [r for r in csv.DictReader(open(results_csv)) if r['site'] == site]
    fmt = lambda v: '$<0.01$' if v < 0.01 else f'${v:.2f}$'
    body = []
    for label in [l for l in FIELD_ORDER if any(r['label'] == l for r in rows)]:
        vals = []
        for kind in ('full', 'best'):
            for det in DETECTORS:
                r = [float(x['pred_in_gt']) for x in rows if x['label'] == label and x['detector'] == det
                     and (kind == 'best' or x['condition'] == 'full')]
                vals.append(fmt(max(r)))
        name = FIELD_LABEL[label] if site == 'site2' else FIELD_LABEL[label].replace(', view 1', '')
        body.append(f'{name} & ' + ' & '.join(vals) + r' \\')
    n = 'AB'[int(site[-1]) - 1]
    if site == 'site1':
        cap = (r'Location~A. Mean detector score inside the traced outline; lower means harder to find. '
               r"``Max'' is the highest detector score over the full frame and crops of $1.5\times$, $2.5\times$ and "
               r'$4\times$ the item size. \texttt{print\_v2} is not pipeline output.')
    else:
        cap = (r'Location~B, same measures as Table~\ref{tab:locA}. The two M81 rows are two photographs '
               r'of one unmoved drape, taken 17\,s apart.')
    return _write(r'''\begin{table}[t]
\centering
\caption{''' + cap + r'''}
\label{tab:loc''' + n + r'''}
\footnotesize
\setlength{\tabcolsep}{4pt}
\begin{tabular}{lrrrr}
\hline
 & \multicolumn{2}{c}{Full frame} & \multicolumn{2}{c}{Max} \\
Item & SINet-V2 & DGNet & SINet-V2 & DGNet \\
\hline
''' + '\n'.join(body) + r'''
\hline
\end{tabular}
\end{table}''', out)


def recolor_table(site_csv, ablation_csv, out=None):
    """Table tab:recolor: how much color alone does (notebook 2, uses the notebook 1 ablation rows).

    recolored = base pattern recolored with the five site colors, no style transfer (nonst_k5);
    all arms share scenes, shapes and positions, so each column is a paired difference."""
    agg = defaultdict(list)
    for f in (site_csv, ablation_csv):
        for r in csv.DictReader(open(f)):
            if r['condition'] in ('site', 'generic', 'nonst_k5', 'kfull'):
                agg[(r['bg'], r['style'], r['detector'], r['shape'], r['condition'])].append(float(r['S_alpha']))
    bgs = sorted({k[0] for k in agg}); shapes = sorted({k[3] for k in agg})
    vec = lambda st, det, c: np.array([np.mean([np.mean(agg[(b, st, det, sh, c)]) for sh in shapes]) for b in bgs])
    cells = [(st, det) for st in STYLES for det in DETECTORS]
    cols = [('nonst_k5', 'generic'), ('site', 'nonst_k5'), ('kfull', 'generic')]
    D = {(c, k): vec(*c, k[0]) - vec(*c, k[1]) for c in cells for k in cols}
    H = {k: dict(zip(cells, _holm([stats.wilcoxon(D[(c, k)]).pvalue for c in cells]))) for k in cols}
    body = []
    for c in cells:
        vals = [f'${D[(c, k)].mean():+.3f}^{{*}}$' if H[k][c] < 0.05 else f'${D[(c, k)].mean():+.3f}$' for k in cols]
        body.append(f"{NAME[c[0]]:8} & {c[1]:8} & " + ' & '.join(vals) + r' \\')
    return _write(r'''\begin{table}[t]
\centering
\caption{What color alone does. Each row is named after the base pattern. Each column is a paired $\Delta S_\alpha$ on the same
33 scenes and positions, given as the mean of the 33 scene-level differences; a negative value means the first pattern is harder to
find. ``Recolored'' is the base pattern recolored with the five site colors, with
no style transfer; ``site'' is the site-specific pattern; ``no limit'' is the
style-transfer pattern colored with no color limit. An asterisk marks a
difference that is significant after Holm correction over the six cells of its
column (two-sided Wilcoxon).}
\label{tab:recolor}
\footnotesize
\setlength{\tabcolsep}{3pt}
\begin{tabular}{llrrr}
\hline
 & & recolored & site & no limit \\
Base & Detector & $-$ base & $-$ recolored & $-$ base \\
\hline
''' + '\n'.join(body) + r'''
\hline
\end{tabular}
\end{table}''', out)

## The two detectors

The two detectors are loaded once, and every step below uses them.

In [ ]:
from cod_metrics import CODMetrics
from detectors import build_detectors
import calibration
sinet, dgnet = build_detectors(DEVICE)

## The two locations (2026-09-18)

**Location A** (`IMG_4398`–`IMG_4404`) is 2.7–5.9 m from the generation frame. The items lie flat on leaf litter in canopy shade, all photographed within 162 s. **Location B** (`IMG_4405`–`IMG_4412`) is 11.3 km away, near most of the test scenes of notebook 2; the fabric is draped over shrubs against green understory. Each location is calibrated with its own ColorChecker photograph (`IMG_4398`, `IMG_4405`). Compare items **within** a location only.

`IMG_4399` is a second photograph of the `IMG_4400` placement, 2 s later, and is not used. `IMG_4408` and `IMG_4409` show one unmoved M81 drape from two camera positions, 17 s apart: their difference measures viewpoint, not placement. `IMG_4412` contains no garment (empty photograph). Every item has one placement only, so no statistics are computed.

In [ ]:
C_DIR = ROOT/'images'/'field_2026-09-18_two_sites'
C_OUT = OUT/'two_sites'; C_OUT.mkdir(parents=True, exist_ok=True)
SITES = {'site1': ('IMG_4398', ['IMG_4399', 'IMG_4400', 'IMG_4401', 'IMG_4402', 'IMG_4403', 'IMG_4404']),
         'site2': ('IMG_4405', ['IMG_4406', 'IMG_4407', 'IMG_4408', 'IMG_4409', 'IMG_4410', 'IMG_4411', 'IMG_4412'])}
ITEMS_C = [('site1', 'IMG_4400', 'print_v2'), ('site1', 'IMG_4401', 'hunting_realtree'), ('site1', 'IMG_4402', 'multicam_ocp'),
           ('site1', 'IMG_4403', 'woodland_m81'), ('site1', 'IMG_4404', 'print_v1'),
           ('site2', 'IMG_4406', 'print_v2'), ('site2', 'IMG_4407', 'print_v1'), ('site2', 'IMG_4408', 'woodland_m81_b'),
           ('site2', 'IMG_4409', 'woodland_m81'), ('site2', 'IMG_4410', 'multicam_ocp'), ('site2', 'IMG_4411', 'hunting_realtree')]
CONTROL = ('site2', 'IMG_4412')
png_c = lambda site, stem: C_DIR/'calibrated'/site/f'{stem}.png'

for site, (chart, frames) in SITES.items():
    other = [f for s2, (c2, fr2) in SITES.items() if s2 != site for f in [c2] + fr2]   # one chart per run
    calibration.calibrate(str(C_DIR.relative_to(ROOT)), chart, str((C_DIR/'calibrated'/site).relative_to(ROOT)), exclude=other)
rows_c = score_items(ITEMS_C, lambda it: png_c(it[0], it[1]),
                     lambda it: C_DIR/'_analysis'/f'gt_{it[0]}_{it[1]}_{it[2]}.png', C_OUT/'overlays.png')
write_csv(rows_c, C_OUT/'results.csv', ['site', 'photo', 'label'])

### Location A and Location B tables

In [ ]:
LABEL = {'print_v1': 'print_v1 (pipeline)', 'print_v2': 'print_v2', 'multicam_ocp': 'MultiCam', 'woodland_m81': 'M81',
         'woodland_m81_b': 'M81, second view', 'hunting_realtree': 'Realtree'}
for site, title in (('site1', 'Location A - generation site, flat on leaf litter'),
                    ('site2', 'Location B - 11.3 km away, draped on shrubs')):
    print(f"\n{title}\n{'item':22}{'full SINet':>11}{'full DGNet':>11}{'best SINet':>11}{'best DGNet':>11}")
    for it in [x for x in ITEMS_C if x[0] == site]:
        (fs, bs), (fd, bd) = full_and_best(rows_c, it, 'SINet-V2'), full_and_best(rows_c, it, 'DGNet')
        print(f'{LABEL[it[2]]:22}{fmt(fs):>11}{fmt(fd):>11}{fmt(bs):>11}{fmt(bd):>11}')

m1, m2 = ('site2', 'IMG_4409', 'woodland_m81'), ('site2', 'IMG_4408', 'woodland_m81_b')
print('\nM81 drape, two camera positions (viewpoint only):')
for d in DETECTORS:
    (f1, b1), (f2, b2) = full_and_best(rows_c, m1, d), full_and_best(rows_c, m2, d)
    print(f'  {d:9} full frame {f1:.2f} vs {f2:.2f} (difference {abs(f1-f2):.2f})   best {b1:.2f} vs {b2:.2f} (difference {abs(b1-b2):.2f})')

### The empty photograph

`IMG_4412` contains no garment, so any response is a false alarm.

In [ ]:
cimg = load_rgb(png_c(*CONTROL))
ctrl = []
fig, ax = plt.subplots(1, 3, figsize=(18, 6)); ax[0].imshow(cimg); ax[0].set_title(f'{CONTROL[1]} (no garment)')
for i, (name, det) in enumerate(zip(DETECTORS, (sinet, dgnet)), start=1):
    pred = det.predict(cimg)
    n, lab_, st, _ = cv2.connectedComponentsWithStats((pred > 0.9).astype(np.uint8))
    largest = int(st[1:, cv2.CC_STAT_AREA].max()) if n > 1 else 0
    ctrl.append(dict(site=CONTROL[0], photo=CONTROL[1], detector=name, pred_max=float(pred.max()), pred_mean=float(pred.mean()),
                     frac_above_0p5=float((pred > 0.5).mean()), frac_above_0p9=float((pred > 0.9).mean()),
                     n_blobs_above_0p9=n - 1, largest_blob_px=largest))
    print(f"{name:9} max {pred.max():.2f}   frame above 0.9: {(pred > 0.9).mean():.2%}   largest blob {largest} px")
    ax[i].imshow(cimg); ax[i].imshow(pred, alpha=0.55, cmap='jet', vmin=0, vmax=1); ax[i].set_title(name)
for a in ax: a.axis('off')
plt.tight_layout(); plt.savefig(C_OUT/'empty_control.png', dpi=80, bbox_inches='tight'); plt.show()
with open(C_OUT/'empty_control.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(ctrl[0])); w.writeheader(); w.writerows(ctrl)

### Relative local contrast — Results, "Printed fabric in the field"

Local contrast is the mean standard deviation of $L^*$ in a 15×15 pixel window. The relative local contrast of an item is the local contrast inside its traced outline divided by the same measure on a ring of ground around it, 0.3 times the item size wide (`tex_ratio` in the CSV file). The cell also writes the color difference to the ring ($\Delta E$, CIE76).

In [ ]:
%%module field_texture_colour
"""Relative local contrast and color difference of each item against the ground around it.

    local contrast        = mean standard deviation of L* in a 15 x 15 pixel window
    relative contrast     = local contrast inside the traced outline (eroded by 21 px) divided by
                            the same measure on a ring of ground around the item, 0.3 times the
                            item size wide (tex_ratio in the CSV)
    color difference (dE) = CIE76 distance between the mean Lab inside and on the ring

The CSV also gives the highest detector score of each item (full frame and best look) from
results.csv. This cell runs as a script and writes field_texture_colour.csv.
"""
import csv
from pathlib import Path
import numpy as np, cv2

ROOT = Path.cwd()          # project root (the notebook starts here)
AN = ROOT / 'images' / 'field_2026-09-18_two_sites' / '_analysis'
PNG = ROOT / 'images' / 'field_2026-09-18_two_sites' / 'calibrated'
OUT = ROOT / 'results' / '03_field_test' / 'two_sites'

ITEMS = [
    ('site1', 'IMG_4400', 'print_v2'), ('site1', 'IMG_4401', 'hunting_realtree'),
    ('site1', 'IMG_4402', 'multicam_ocp'), ('site1', 'IMG_4403', 'woodland_m81'),
    ('site1', 'IMG_4404', 'print_v1'),
    ('site2', 'IMG_4406', 'print_v2'), ('site2', 'IMG_4407', 'print_v1'),
    ('site2', 'IMG_4408', 'woodland_m81_b'), ('site2', 'IMG_4409', 'woodland_m81'),
    ('site2', 'IMG_4410', 'multicam_ocp'), ('site2', 'IMG_4411', 'hunting_realtree'),
]


def lab(bgr):
    l = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB).astype(np.float64)
    return np.stack([l[..., 0] * 100 / 255., l[..., 1] - 128, l[..., 2] - 128], -1)


def local_std(L, k=15):
    mu, mu2 = cv2.blur(L, (k, k)), cv2.blur(L * L, (k, k))
    return np.sqrt(np.maximum(mu2 - mu * mu, 0))


det_full, det_best = {}, {}
res = OUT / 'results.csv'
if res.exists():
    for r in csv.DictReader(open(res)):
        k = (r['site'], r['photo'], r['detector'])
        v = float(r['pred_in_gt'])
        if r['condition'] == 'full':
            det_full[k] = v
        det_best[k] = max(det_best.get(k, 0.0), v)

rows = []
for site, stem, label in ITEMS:
    img = cv2.imread(str(PNG / site / f'{stem}.png'))
    m = cv2.imread(str(AN / f'gt_{site}_{stem}_{label}.png'), 0) > 127
    LAB = lab(img)
    sd = local_std(LAB[..., 0])
    inner = cv2.erode(m.astype(np.uint8), np.ones((21, 21), np.uint8)).astype(bool)
    ys, xs = np.where(m)
    side = max(ys.max() - ys.min(), xs.max() - xs.min())
    k = max(int(side * 0.6) // 2 * 2 + 1, 3)
    ring = cv2.dilate(m.astype(np.uint8), np.ones((k, k), np.uint8)).astype(bool) & ~m
    tex, ring_tex = sd[inner].mean(), sd[ring].mean()
    fab_lab, bg_lab = LAB[inner].mean(0), LAB[ring].mean(0)
    d = fab_lab - bg_lab
    ff = max(det_full.get((site, stem, dd), 0.0) for dd in ('SINet-V2', 'DGNet')) if det_full else float('nan')
    bb = max(det_best.get((site, stem, dd), 0.0) for dd in ('SINet-V2', 'DGNet')) if det_best else float('nan')
    rows.append(dict(site=site, photo=stem, label=label, mask_px=int(m.sum()),
                     tex=round(tex, 2), ring_tex=round(ring_tex, 2), tex_ratio=round(tex / ring_tex, 3),
                     dL=round(d[0], 1), da=round(d[1], 1), db=round(d[2], 1),
                     colour_dE=round(float(np.linalg.norm(d)), 1),
                     full_frame_max_pred_in=round(ff, 3), best_max_pred_in=round(bb, 3)))

with open(OUT / 'field_texture_colour.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

for site in ('site1', 'site2'):
    sub = [r for r in rows if r['site'] == site]
    print(f"\n{site}  ({'flat on brown litter' if site == 'site1' else 'draped, green understory'})")
    print(f"  {'garment':17s}{'tex':>6s}{'ring':>6s}{'ratio':>7s}{'dL*':>7s}{'da*':>7s}{'db*':>7s}{'colour dE':>11s}{'full':>7s}{'best':>7s}")
    for r in sorted(sub, key=lambda r: r['colour_dE']):
        print(f"  {r['label']:17s}{r['tex']:6.2f}{r['ring_tex']:6.2f}{r['tex_ratio']:7.3f}"
              f"{r['dL']:+7.1f}{r['da']:+7.1f}{r['db']:+7.1f}{r['colour_dE']:11.1f}"
              f"{r['full_frame_max_pred_in']:7.2f}{r['best_max_pred_in']:7.2f}")

print('\nprint generations, background match head to head:')
for site in ('site1', 'site2'):
    v1 = next(r for r in rows if r['site'] == site and r['label'] == 'print_v1')
    v2 = next(r for r in rows if r['site'] == site and r['label'] == 'print_v2')
    print(f"  {site}  texture ratio v1={v1['tex_ratio']:.3f} v2={v2['tex_ratio']:.3f}"
          f"   colour dE v1={v1['colour_dE']:.1f} v2={v2['colour_dE']:.1f}")
print('\n->', OUT / 'field_texture_colour.csv')

## Figure `fig_field` for the paper

The pipeline sheet `print_v1` at each location: the photograph with its traced outline in red, and the full-frame output of each detector (white = strong response) with its mean inside the outline.

In [ ]:
det = dict(zip(DETECTORS, (sinet, dgnet)))
items = [('Location A', 'site1', 'IMG_4404'), ('Location B', 'site2', 'IMG_4407')]   # print_v1 at each location
FIGDIR = OUT/'figures'; FIGDIR.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(2, 3, figsize=(7.2, 6.2))
for r, (name, site, stem) in enumerate(items):
    img = load_rgb(png_c(site, stem))
    gt = cv2.imread(str(C_DIR/'_analysis'/f'gt_{site}_{stem}_print_v1.png'), 0)
    ax[r, 0].imshow(img)
    for c in cv2.findContours(gt, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]:
        ax[r, 0].plot(c[:, 0, 0], c[:, 0, 1], 'r-', lw=1.2)
    ax[r, 0].set_title(f'{name}: print_v1 (red outline)', fontsize=8)
    for k, (dn, d) in enumerate(det.items(), start=1):
        pred = d.predict(img)
        ax[r, k].imshow(pred, cmap='gray', vmin=0, vmax=1)
        for c in cv2.findContours(gt, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]:
            ax[r, k].plot(c[:, 0, 0], c[:, 0, 1], 'r-', lw=0.8)
        ax[r, k].set_title(f'{dn}: mean inside {pred[gt > 127].mean():.2f}', fontsize=8)
for a in ax.ravel(): a.axis('off')
plt.tight_layout(); plt.savefig(FIGDIR/'fig_field.png', dpi=200, bbox_inches='tight'); plt.show()
print('->', FIGDIR/'fig_field.png')

## Tables `tab:locA` and `tab:locB` for the paper

The cell writes the LaTeX of the tables to `results/03_field_test/tables/`.

In [ ]:
import paper_tables as pt
TAB = OUT/'tables'
for site, loc in (('site1', 'A'), ('site2', 'B')):
    print(pt.location_table(C_OUT/'results.csv', site, out=TAB/f'tab_loc{loc}.tex')); print()

## Answer — does the advantage survive printing?

This cell applies the rule from the top of the notebook. "LESS found" means that the printed pipeline sheet had a lower detector score than the M81 garment by more than the viewpoint difference. That is the direction notebook 2 found in software.

In [ ]:
MEAS = ('full frame', 'best')
spread = {(d, k): abs(full_and_best(rows_c, m1, d)[k] - full_and_best(rows_c, m2, d)[k]) for d in DETECTORS for k in (0, 1)}
BASE = {'site1': [('site1', 'IMG_4403', 'woodland_m81')], 'site2': [m1, m2]}
print('PHYSICAL TEST - print_v1 (site-specific M81, printed) against the M81 garment (its base)')
verdicts = {}
for site, bases in BASE.items():
    v1 = next(it for it in ITEMS_C if it[0] == site and it[2] == 'print_v1')
    print(f"  Location {'AB'[int(site[-1]) - 1]}")
    for d in DETECTORS:
        for k in (0, 1):
            s = full_and_best(rows_c, v1, d)[k]; b = float(np.mean([full_and_best(rows_c, it, d)[k] for it in bases]))
            diff, lim = s - b, spread[(d, k)]
            v = 'LESS found' if diff < -lim else ('MORE found' if diff > lim else 'within viewpoint difference')
            verdicts.setdefault(site, []).append(v)
            print(f"    {d:9} {MEAS[k]:10} sheet {s:.2f}  garment {b:.2f}  difference {diff:+.2f}  (viewpoint {lim:.2f})  -> {v}")
for site, vs in verdicts.items():
    n_less, n_more = vs.count('LESS found'), vs.count('MORE found')
    print(f"ANSWER Location {'AB'[int(site[-1]) - 1]}: sheet less found in {n_less} of {len(vs)} comparisons, more found in {n_more}, "
          f"no clear difference in {len(vs) - n_less - n_more}.")
mixed = any(v.count('MORE found') for v in verdicts.values())
print('ANSWER overall: ' + ('the results are mixed, so transfer to cloth is neither confirmed nor refuted by this study.' if mixed
      else 'the printed sheet was never found more than its base garment, but with one placement per item this is an observation, not a test.'))

## Appendix — the code that made `print_v2`

### `print_v2` — Methodology, "Fabrication"

The low contrast inside `print_v1` motivated a second, contrast-enhanced sheet. Every pixel of `print_v1` is moved to one of the same five colors: its target lightness is its current $L^*$ plus a scaled noise field (white noise, Gaussian-filtered at 3, 6, 11 and 20 mm on the printed sheet), and a cost bias for each color keeps each color's share of the sheet. It is not style-transfer output.

This notebook defines the code but does not start it. `print_v2_revision.make_print_v2()` makes the sheet again (about 3 minutes) and writes it to `results/03_field_test/print_v2_revision/`; a test on 2026-09-23 found 99.3 % of its pixels equal to the file that went to the printer.

In [ ]:
%%module print_v2_revision
"""print_v2: the contrast-enhanced copy of print_v1 (Methodology, "Fabrication").

Every pixel of print_v1 is moved to one of the same five colors. The target lightness of a
pixel is its current L* plus a scaled noise field (white noise, Gaussian-filtered at 3, 6, 11
and 20 mm on the printed sheet). Each color also gets a cost bias, tuned so that each color
keeps its share of the sheet. The noise gain with the most local contrast wins. This is not
style-transfer output.

make_print_v2() writes the sheet at 200 dpi, tuning.json and candidates.json to OUT.
"""
import json
import os

import cv2
import numpy as np
from PIL import Image

ROOT = os.getcwd()   # project root (the notebook starts here)
# print_v1: the non-repeating M81 sheet. The file that went to the printer was not kept;
# this is the file of 2026-09-02, from which print_v2 is made again in 99.3 % of pixels
SRC = f'{ROOT}/images/printed_sheets/print_v1_woodland_unique_sheet_28.16x27.52in_200dpi.png'
OUT = f'{ROOT}/results/03_field_test/print_v2_revision'
DPI = 200
IN_W = 28.16                       # printed sheet width in inches
AB_W = 1.0                         # weight of the a*b* distance against the L* distance


def srgb_to_lin(x): return np.where(x <= 0.04045, x/12.92, ((x+0.055)/1.055)**2.4)
def lin_to_srgb(x): return np.where(x <= 0.0031308, 12.92*x, 1.055*x**(1/2.4)-0.055)


def resize_ll(img, w):
    """Resize in linear light to width w."""
    lin = srgb_to_lin(img.astype(np.float32)/255.)
    h = int(round(img.shape[0]*w/img.shape[1]))
    return (np.clip(lin_to_srgb(cv2.resize(lin, (w, h), interpolation=cv2.INTER_AREA)), 0, 1)*255).astype(np.uint8)


def bgr_lab(bgr):
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB).astype(np.float32); lab[..., 0] *= 100/255.
    return lab


def tex(bgr, k=15):
    """Local contrast: mean standard deviation of L* in a k x k window."""
    L = bgr_lab(bgr)[..., 0]
    mu = cv2.blur(L, (k, k)); mu2 = cv2.blur(L*L, (k, k))
    return float(np.sqrt(np.maximum(mu2-mu*mu, 0)).mean())


def noise_field(shape, scale_px, px_per_mm, seed=11):
    rng = np.random.default_rng(seed)
    N = np.zeros(shape, np.float32)
    for mm, wgt in [(3, 0.8), (6, 1.2), (11, 1.0), (20, 0.6)]:
        s = mm*px_per_mm*scale_px
        n = cv2.GaussianBlur(rng.standard_normal(shape).astype(np.float32), (0, 0), max(s, 0.8))
        N += wgt*n/max(n.std(), 1e-9)
    return N/N.std()


def assign(idx, noise, gain, plab, biases):
    """Color index of each pixel: the nearest palette color to (L* + gain x noise, a*, b*), plus the bias."""
    src = plab[idx]
    tL = src[..., 0] + gain*noise
    cost = np.stack([(tL-plab[j, 0])**2 + AB_W*((src[..., 1]-plab[j, 1])**2 + (src[..., 2]-plab[j, 2])**2) + biases[j]
                     for j in range(len(plab))], -1)
    return cost.argmin(-1).astype(np.uint8)


def fit_biases(idx_w, noise_w, gain, plab, target, iters=60, eta=900.):
    """Tune the biases until each color has its target share of the pixels."""
    b = np.zeros(len(plab), np.float32)
    for _ in range(iters):
        a = assign(idx_w, noise_w, gain, plab, b)
        f = np.bincount(a.ravel(), minlength=len(plab))/a.size
        b += eta*(f-target)
        b -= b.min()
    return b, f


def make_print_v2(gains=(20, 30, 40, 55, 75, 100)):
    os.makedirs(OUT, exist_ok=True)
    full = cv2.imread(SRC)
    H, W = full.shape[:2]
    px_per_mm = (W/IN_W)/25.4
    print(f'source {W}x{H} @ {DPI} DPI = {IN_W:.2f}x{H/DPI:.2f} in; {px_per_mm:.2f} px/mm')
    pal, inv = np.unique(full.reshape(-1, 3), axis=0, return_inverse=True)
    idx_full = inv.reshape(H, W).astype(np.uint8)
    pal_lab = bgr_lab(pal[None].astype(np.uint8))[0].astype(np.float32)
    target = np.bincount(inv, minlength=len(pal))/inv.size

    # tune the gain on a quarter-size copy
    work = resize_ll(full, W//4)
    d = ((work.reshape(-1, 1, 3).astype(np.float32)-pal.reshape(1, -1, 3))**2).sum(-1)
    idx_w = d.argmin(1).reshape(work.shape[:2]).astype(np.uint8)
    N_w = noise_field(idx_w.shape, 0.25, px_per_mm)
    best = None
    for g in gains:
        b, f = fit_biases(idx_w, N_w, g, pal_lab, target)
        a = assign(idx_w, N_w, g, pal_lab, b)
        t600 = tex(resize_ll(pal[a], 600))
        dfrac = float(np.abs(np.bincount(a.ravel(), minlength=len(pal_lab))/a.size - target).max())
        dL = float((pal_lab[a][..., 0]).mean() - (pal_lab[idx_w][..., 0]).mean())
        print(f'  gain={g:5.1f} tex@600={t600:5.2f} maxΔfrac={dfrac:.3f} ΔmeanL*={dL:+.2f}', flush=True)
        if best is None or t600 > best[1]:
            best = (g, t600, b, dfrac, dL)
    json.dump({'px_per_mm': px_per_mm,
               'v2': {'gain': float(best[0]), 'tex600': float(best[1]), 'max_dfrac': best[3], 'dmeanL': best[4]}},
              open(f'{OUT}/tuning.json', 'w'), indent=1)

    # the full-size sheet with the best gain
    N_f = cv2.resize(N_w, (W, H), interpolation=cv2.INTER_CUBIC); N_f /= N_f.std()
    out = np.empty_like(full)
    for y0 in range(0, H, 512):
        y1 = min(y0+512, H)
        out[y0:y1] = pal[assign(idx_full[y0:y1], N_f[y0:y1], best[0], pal_lab, best[2])]
    name = 'victoria_woodland_texfix_v2_quota'
    Image.fromarray(cv2.cvtColor(out, cv2.COLOR_BGR2RGB)).save(f'{OUT}/{name}_{IN_W:.2f}x{H/DPI:.2f}in_{DPI}dpi.png', dpi=(DPI, DPI))

    def summary(img, name):
        d2 = ((img.reshape(-1, 1, 3).astype(np.float32)-pal.reshape(1, -1, 3))**2).sum(-1).argmin(1)
        return dict(name=name, colours=int(len(np.unique(img.reshape(-1, 3), axis=0))), tex600=tex(resize_ll(img, 600)),
                    tex600_blur=tex(resize_ll(cv2.GaussianBlur(img, (0, 0), 0.5*px_per_mm), 600)),   # 0.5 mm ink spread
                    tex208=tex(resize_ll(img, 208)),
                    fractions=[float(x) for x in np.bincount(d2, minlength=len(pal))/d2.size],
                    meanL=float(bgr_lab(img)[..., 0].mean()))
    cands = [summary(full, 'original_design'), summary(out, name)]
    for c in cands:
        print(f"{c['name']}: {c['colours']} colours, local contrast at 600 px {c['tex600']:.2f}, mean L* {c['meanL']:.1f}")
    json.dump(cands, open(f'{OUT}/candidates.json', 'w'), indent=1)
    return f'{OUT}/{name}_{IN_W:.2f}x{H/DPI:.2f}in_{DPI}dpi.png'